In [69]:
import time
import io
import zipfile
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from functools import lru_cache
from tqdm import tqdm


# =============================
# 0) KRX 상장사 목록 (pykrx 없이)
# =============================
def read_krx_code(exclude_spac=True):
    url = "http://kind.krx.co.kr/corpgeneral/corpList.do?method=download&searchType=13"
    krx = pd.read_html(url, header=0, encoding="euc-kr")[0]
    krx = krx[["종목코드", "회사명"]].rename(columns={"종목코드": "code", "회사명": "company"})
    krx["code"] = krx["code"].astype(str).str.zfill(6)
    if exclude_spac:
        krx = krx[~krx["company"].str.contains("스펙", na=False)].reset_index(drop=True)
    return krx


# =============================
# 1) DART 설정
# =============================
DART_BASE = "https://opendart.fss.or.kr/api"

REPRT_TO_QEND_MMDD = {
    "11011": "03-31",
    "11012": "06-30",
    "11013": "09-30",
    "11014": "12-31",
}

def _to_int_or_none(x):
    if x is None:
        return None
    s = str(x).replace(",", "").strip()
    if s in ("", "-", "—"):
        return None
    try:
        return int(s)
    except ValueError:
        return None


# =============================
# 2) corpCode.xml -> stock_code -> corp_code
# =============================
@lru_cache(maxsize=1)
def load_corpcode_map(api_key: str):
    url = f"{DART_BASE}/corpCode.xml"
    r = requests.get(url, params={"crtfc_key": api_key}, timeout=60)
    r.raise_for_status()

    z = zipfile.ZipFile(io.BytesIO(r.content))
    xml_name = next((n for n in z.namelist() if n.upper().endswith(".XML")), None)
    if not xml_name:
        raise RuntimeError("corpCode.xml(zip)에서 XML 파일을 찾지 못했습니다.")

    root = ET.fromstring(z.read(xml_name))

    mapping = {}
    for el in root.findall("list"):
        corp_code = (el.findtext("corp_code") or "").strip()
        stock_code = (el.findtext("stock_code") or "").strip()
        if corp_code and stock_code:
            mapping[stock_code.zfill(6)] = corp_code
    if not mapping:
        raise RuntimeError("corpCode.xml 파싱 결과가 비었습니다.")
    return mapping


# =============================
# 3) stockTotqySttus.json 파싱 (핵심 수정)
# =============================
def fetch_shares_from_stockTotqy(api_key: str, corp_code: str, bsns_year: int, reprt_code: str):
    """
    stockTotqySttus.json에서
      total_shares: now_to_isu_stock_totqy (현재까지 발행한 주식의 총수) 우선
                   없으면 isu_stock_totqy(발행할 주식의 총수) fallback
      float_shares: distb_stock_co
    반환
    """
    url = f"{DART_BASE}/stockTotqySttus.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(bsns_year),
        "reprt_code": reprt_code,
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    if data.get("status") != "000":
        return None, None, data.get("status"), data.get("message")

    rows = data.get("list", []) or []
    if not rows:
        return None, None, "EMPTY", "no list"

    # '합계' 행 우선
    target = None
    for row in rows:
        if (row.get("se") or "").strip() == "합계":
            target = row
            break
    if target is None:
        target = rows[0]

    # ✅ 핵심: issued shares에 가까운 필드 우선 사용
    total_sh = _to_int_or_none(target.get("now_to_isu_stock_totqy"))
    if total_sh is None:
        total_sh = _to_int_or_none(target.get("isu_stock_totqy"))

    float_sh = _to_int_or_none(target.get("distb_stock_co"))

    return total_sh, float_sh, "000", "OK"


# =============================
# 4) reprt_code 여러개를 던져서 하나라도 값이 있으면 채택
# =============================
def fetch_shares_best_effort(api_key: str, corp_code: str, bsns_year: int, reprt_codes):
    """
    reprt_codes 순서대로 호출해서 total/float 중 하나라도 채워지면 채택.
    """
    best_total = None
    best_float = None
    last_status = None
    last_msg = None

    for rc in reprt_codes:
        total_sh, float_sh, st, msg = fetch_shares_from_stockTotqy(api_key, corp_code, bsns_year, rc)
        last_status, last_msg = st, msg

        if best_total is None and total_sh is not None:
            best_total = total_sh
        if best_float is None and float_sh is not None:
            best_float = float_sh

        # 둘 다 확보되면 종료
        if best_total is not None and best_float is not None:
            return best_total, best_float, rc, "000", "OK"

    return best_total, best_float, None, last_status, last_msg


# =============================
# 5) 수집 -> DF 2개 + merge
# =============================
def build_total_float_and_final_dfs_dart_only(
    api_key: str,
    tickers,
    years,
    reprt_codes=("11014", "11011", "11012", "11013"),  # ✅ 추천 순서(사업보고서 우선)
    sleep_sec=0.12,
    verbose=True,
):
    if isinstance(years, (list, tuple, set)):
        years = [int(y) for y in years]
    else:
        years = [int(years)]

    tickers = [str(t).zfill(6) for t in list(tickers)]

    if isinstance(reprt_codes, str):
        raise ValueError("reprt_codes는 문자열이 아니라 튜플/리스트로 넣어야 합니다. 예: ('11014',)")
    reprt_codes = list(reprt_codes)

    corp_map = load_corpcode_map(api_key)

    total_rows = []
    float_rows = []

    total_jobs = len(tickers) * len(years)
    done = 0

    # 디버깅용 카운트
    cnt_total = 0
    cnt_float = 0
    cnt_corp_missing = 0
    cnt_all_missing = 0

    for y in years:
        for tkr in tickers:
            done += 1
            corp_code = corp_map.get(tkr)
            if not corp_code:
                cnt_corp_missing += 1
                continue

            total_sh, float_sh, used_rc, st, msg = fetch_shares_best_effort(
                api_key=api_key,
                corp_code=corp_code,
                bsns_year=y,
                reprt_codes=reprt_codes
            )

            # date는 "used_rc" 기준으로 분기말로 찍되, used_rc 없으면 사업연도말로 찍음
            rc_for_date = used_rc if used_rc in REPRT_TO_QEND_MMDD else "11014"
            q_end_dt = pd.to_datetime(f"{y}-{REPRT_TO_QEND_MMDD[rc_for_date]}")

            if total_sh is not None:
                total_rows.append((q_end_dt, tkr, total_sh))
                cnt_total += 1
            if float_sh is not None:
                float_rows.append((q_end_dt, tkr, float_sh))
                cnt_float += 1

            if total_sh is None and float_sh is None:
                cnt_all_missing += 1

            if verbose and (done % 500 == 0 or done == total_jobs):
                print(f"[PROGRESS] {done:,}/{total_jobs:,} ({done/total_jobs:.1%}) "
                      f"| total_ok={cnt_total:,}, float_ok={cnt_float:,}, all_missing={cnt_all_missing:,}, corp_missing={cnt_corp_missing:,}")

            time.sleep(sleep_sec)

    df_total = pd.DataFrame(total_rows, columns=["date", "ticker", "total_shares"]) \
                .sort_values(["ticker", "date"]).reset_index(drop=True)

    df_float = pd.DataFrame(float_rows, columns=["date", "ticker", "float_shares"]) \
                .sort_values(["ticker", "date"]).reset_index(drop=True)

    df_merged = pd.merge(df_total, df_float, on=["date", "ticker"], how="left")
    df_merged["shares_final"] = df_merged["float_shares"].fillna(df_merged["total_shares"])
    df_merged = df_merged.sort_values(["ticker", "date"]).reset_index(drop=True)

    return df_total, df_float, df_merged


# =============================
# 6) 전 종목 래퍼
# =============================
def build_all_dfs_dart_only_no_pykrx(
    api_key: str,
    start_year: int,
    end_year: int,
    reprt_codes=("11014", "11011", "11012", "11013"),
    sleep_sec=0.12,
    verbose=True,
    exclude_spac=True,
):
    krx = read_krx_code(exclude_spac=exclude_spac)
    tickers = krx["code"].tolist()
    years = list(range(int(start_year), int(end_year) + 1))

    return build_total_float_and_final_dfs_dart_only(
        api_key=api_key,
        tickers=tickers,
        years=years,
        reprt_codes=reprt_codes,
        sleep_sec=sleep_sec,
        verbose=verbose,
    )


def summarize_coverage(df_total: pd.DataFrame, df_float: pd.DataFrame, df_final: pd.DataFrame, tickers_all):
    """
    전 종목 대비 커버리지 요약
    - tickers_all: 전체 ticker 리스트 (6자리 문자열)
    """
    tickers_all = [str(t).zfill(6) for t in tickers_all]
    n_all = len(set(tickers_all))

    got_total = set(df_total["ticker"].astype(str).str.zfill(6).unique()) if not df_total.empty else set()
    got_float = set(df_float["ticker"].astype(str).str.zfill(6).unique()) if not df_float.empty else set()
    got_final = set(df_final["ticker"].astype(str).str.zfill(6).unique()) if not df_final.empty else set()

    n_total = len(got_total)
    n_float = len(got_float)
    n_final = len(got_final)

    # final이란 결국 total 또는 float가 있다는 뜻이므로, final 커버리지가 핵심
    pct_total = (n_total / n_all * 100) if n_all else 0
    pct_float = (n_float / n_all * 100) if n_all else 0
    pct_final = (n_final / n_all * 100) if n_all else 0

    missing_final = sorted(list(set(tickers_all) - got_final))
    missing_total = sorted(list(set(tickers_all) - got_total))
    missing_float = sorted(list(set(tickers_all) - got_float))

    summary = {
        "n_all": n_all,
        "n_total": n_total,
        "n_float": n_float,
        "n_final": n_final,
        "pct_total": pct_total,
        "pct_float": pct_float,
        "pct_final": pct_final,
        "missing_final": missing_final,
        "missing_total": missing_total,
        "missing_float": missing_float,
    }
    return summary

In [70]:
API_KEY = "50424484a46daa88b34fcf875f40ca12b79e1fc1"   # ← 본인 키로 교체하세요


# 0) 전 종목 ticker 만들기 (pykrx 없이)
krx = read_krx_code(exclude_spac=True)
all_tickers = krx["code"].tolist()

# 1) 전 종목 수집 (예: 2025년, 보고서코드 우선순위)
df_total_all, df_float_all, df_final_all = build_total_float_and_final_dfs_dart_only(
    api_key=API_KEY,
    tickers=all_tickers,
    years=[2025],
    reprt_codes=("11014","11011","11012","11013"),  # 사업보고서 우선
    sleep_sec=0.15,
    verbose=True
)

# 2) 커버리지 집계
cov = summarize_coverage(df_total_all, df_float_all, df_final_all, all_tickers)

print("\n===== COVERAGE SUMMARY =====")
print(f"전체 종목 수         : {cov['n_all']:,}")
print(f"total_shares 보유    : {cov['n_total']:,}  ({cov['pct_total']:.2f}%)")
print(f"float_shares 보유    : {cov['n_float']:,}  ({cov['pct_float']:.2f}%)")
print(f"shares_final 보유    : {cov['n_final']:,}  ({cov['pct_final']:.2f}%)")

print("\n[참고] shares_final 미존재 ticker 예시(앞 20개):")
print(cov["missing_final"][:20])

# 필요하면 미존재 ticker를 DF로 저장
df_missing_final = pd.DataFrame({"ticker": cov["missing_final"]})

[PROGRESS] 500/2,788 (17.9%) | total_ok=443, float_ok=443, all_missing=57, corp_missing=0
[PROGRESS] 1,000/2,788 (35.9%) | total_ok=904, float_ok=904, all_missing=96, corp_missing=0
[PROGRESS] 1,500/2,788 (53.8%) | total_ok=1,360, float_ok=1,360, all_missing=140, corp_missing=0
[PROGRESS] 2,000/2,788 (71.7%) | total_ok=1,857, float_ok=1,857, all_missing=143, corp_missing=0
[PROGRESS] 2,500/2,788 (89.7%) | total_ok=2,357, float_ok=2,357, all_missing=143, corp_missing=0
[PROGRESS] 2,788/2,788 (100.0%) | total_ok=2,645, float_ok=2,645, all_missing=143, corp_missing=0

===== COVERAGE SUMMARY =====
전체 종목 수         : 2,788
total_shares 보유    : 2,645  (94.87%)
float_shares 보유    : 2,645  (94.87%)
shares_final 보유    : 2,645  (94.87%)

[참고] shares_final 미존재 ticker 예시(앞 20개):
['0007C0', '0009K0', '0013V0', '0015N0', '0015S0', '0054V0', '0068Y0', '0088D0', '0091W0', '0093G0', '0096B0', '0096D0', '0097F0', '0098T0', '0099W0', '0099X0', '0101C0', '0105P0', '0120G0', '0126Z0']


In [75]:
df_final_all[df_final_all['ticker'] == '000660']

,date,ticker,total_shares,float_shares,shares_final
32,2025-12-31,000660,5721980209,690455268,690455268


In [11]:
import pymysql
from DATA.stock_invest_function import get_db_host  # 사용자 정의 함수


def fetch_close_by_code(db_info, code):
    sql = """
        SELECT
            date,
            close
        FROM KSE_Price
        WHERE code = %s
        ORDER BY date
    """

    conn = pymysql.connect(
        host=db_info["host"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_info["database"],   # ✅ 수정 포인트
        port=db_info.get("port", 3307),
        charset="utf8mb4"
    )

    try:
        df = pd.read_sql(sql, conn, params=[code])
    finally:
        conn.close()

    return df

In [12]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'}

df_close = fetch_close_by_code(db_info, "039560")

df_close

,date,close
0,2013-05-21,5750
1,2013-05-22,5880
2,2013-05-23,5690
3,2013-05-24,5620
4,2013-05-27,5870
...,...,...
3092,2025-12-23,3095
3093,2025-12-24,3060
3094,2025-12-26,3040
3095,2025-12-29,3185


In [16]:
df_mktcap = fetch_marketcap_by_ticker(db_info, "005930")
df_mktcap.tail()

,date,marketcap
5126,2025-12-12,6.446490e+14
5127,2025-12-15,6.203780e+14
5128,2025-12-16,6.085390e+14
5129,2025-12-17,6.387290e+14
5130,2025-12-18,6.369530e+14


In [ ]:
# API_KEY = "50424484a46daa88b34fcf875f40ca12b79e1fc1"   # ← 본인 키로 교체하세요
#
# # 예: 2024년 4개 reprt_code를 모두 수집
# df_mktcap = build_daily_float_mktcap_df(
#     api_key=API_KEY,
#     start="20251101",
#     end="20260105",
#     years=[2025],  # 2023년만 쓸 거면 이렇게
#     reprt_codes=("11014"),
#     sleep_sec_dart=0.12,
# )
#
# print(df_mktcap.head())
# print(df_mktcap.tail())
# print(df_mktcap.shape)

C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\bs4\__init__.py:339: UserWarning: You provided Unicode markup but also provided a value for from_encoding. Your from_encoding will be ignored.
  warnings.warn(
Fetching stock data:  18%|█▊        | 495/2789 [00:12<00:53, 42.66it/s]

In [38]:

DART_BASE = "https://opendart.fss.or.kr/api"

def debug_stock_totqy(api_key, corp_code, bsns_year, reprt_code):
    url = f"{DART_BASE}/stockTotqySttus.json"
    params = {
        "crtfc_key": api_key,
        "corp_code": corp_code,
        "bsns_year": str(bsns_year),
        "reprt_code": reprt_code,
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    print("[status]", data.get("status"), "[message]", data.get("message"))
    rows = data.get("list", []) or []
    print("[rows]", len(rows))
    for i, row in enumerate(rows[:20]):  # 앞 20개만
        print(i, {
            "se": row.get("se"),
            "distb_stock_co": row.get("distb_stock_co"),
            "istc_totqy": row.get("istc_totqy"),
            "stock_knd": row.get("stock_knd") or row.get("stock_ty") or row.get("stk_knd"),
        })
    return data

In [43]:
corp_map = load_corpcode_map(API_KEY)
corp_code = corp_map.get("005930")
print("corp_code:", corp_code)

debug_stock_totqy(API_KEY, corp_code, 2025, "11013")

corp_code: 00126380
[status] 000 [message] 정상
[rows] 2
0 {'se': '합계', 'distb_stock_co': '-', 'istc_totqy': '-', 'stock_knd': None}
1 {'se': '비고', 'distb_stock_co': '-', 'istc_totqy': '-', 'stock_knd': None}


{'status': '000',
 'message': '정상',
 'list': [{'rcept_no': '20250515001922',
   'corp_cls': 'Y',
   'corp_code': '00126380',
   'corp_name': '삼성전자',
   'se': '합계',
   'isu_stock_totqy': '-',
   'now_to_isu_stock_totqy': '-',
   'now_to_dcrs_stock_totqy': '-',
   'redc': '-',
   'profit_incnr': '-',
   'rdmstk_repy': '-',
   'etc': '-',
   'istc_totqy': '-',
   'tesstk_co': '-',
   'distb_stock_co': '-',
   'stlm_dt': '2025-03-31'},
  {'rcept_no': '20250515001922',
   'corp_cls': 'Y',
   'corp_code': '00126380',
   'corp_name': '삼성전자',
   'se': '비고',
   'isu_stock_totqy': '-',
   'now_to_isu_stock_totqy': '-',
   'now_to_dcrs_stock_totqy': '-',
   'redc': '-',
   'profit_incnr': '-',
   'rdmstk_repy': '-',
   'etc': '-',
   'istc_totqy': '-',
   'tesstk_co': '-',
   'distb_stock_co': '-',
   'stlm_dt': '2025-03-31'}]}